In [ ]:
from pathlib import Path
import torch
from transformers import AutoTokenizer

## Data collection

In [ ]:
# data collection

def collect_path(folder_path):
    '''collect all the .txt files from certain directory
    input: folder_path
    output: a list of paths of .txt files'''

    # convert the input into a Path object
    folder_path = Path(folder_path)

    # find all .txt files in the directory
    paths = sorted(folder_path.glob("*.txt"))

    return paths

# directories for pos and neg files from train and test folders
train_pos_paths = collect_path("data/aclImdb/train/pos")
train_neg_paths = collect_path("data/aclImdb/train/neg")
test_pos_paths = collect_path("data/aclImdb/test/pos")
test_neg_paths = collect_path("data/aclImdb/test/neg")

print("train_pos_paths:", len(train_pos_paths))
print("train_neg_paths:", len(train_neg_paths))
print("test_pos_paths:", len(test_pos_paths))
print("test_neg_paths:", len(test_neg_paths))

In [ ]:
# read texts and create labels

def extract_data(paths, label):
    '''read texts from a list of file paths and assign them labels
    input:
        paths: a list of text file paths
        label: int, 0 for negative, 1 for positive
    output:
        texts: a list of review texts
        labels: a list of labels related to texts'''
    
    texts = []
    labels = []

    # loop through each file path, read text and related label to their lists
    for path in paths:
        with open(path, 'r', encoding='utf-8')as f:
            text = f.read()
            texts.append(text)
            labels.append(label)
    
    return texts, labels

# read text files and get related labels
train_pos_texts, train_pos_labels = extract_data(train_pos_paths, 1)
train_neg_texts, train_neg_labels = extract_data(train_neg_paths, 0)
test_pos_texts, test_pos_labels = extract_data(test_pos_paths, 1)
test_neg_texts, test_neg_labels = extract_data(test_neg_paths, 0)

train_texts = train_pos_texts + train_neg_texts
train_labels = train_pos_labels + train_neg_labels
test_texts = test_pos_texts + test_neg_texts
test_labels = test_pos_labels + test_neg_labels

print("train_texts:", len(train_texts))
print("train_labels:", len(train_labels))
print("test_texts:", len(test_texts))
print("test_labels:", len(test_labels))

## Data Preprocessing - for BERT and DistilBERT


In [ ]:
# tokenize
from transformers import AutoTokenizer

# load the tokenizer for the BERT model
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# tokenize text data with tokenizer
train_encodings = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=512
)
test_encodings = tokenizer(
    test_texts,
    truncation=True,
    padding=True,
    max_length=512
)

print(len(train_encodings["input_ids"]))
print(len(train_encodings["attention_mask"]))
print(len(train_encodings["token_type_ids"]))

In [ ]:
# datasets
from torch.utils.data import Dataset

# define a custom dataset class
class IMDBDataset(Dataset):
    
    def __init__(self, encodings, labels):
        '''initialize the dataset with encoded texts and labels
        input: tokenized outputs
        labels: sentiment labels corresponding to each encoded text'''

        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        '''return the number of samples in the dataset'''
        return len(self.labels)

    def __getitem__(self, idx):
        '''create one sample from the dataset at index idx'''
        # create an empty dictionary to store the sample 
        item = {}

        # loop through each pair in encodings
        for key, value in self.encodings.items():
            item[key] = torch.tensor(value[idx])  # take the idx-th element and convert it to a tensor
        item["labels"] = torch.tensor(self.labels[idx])  # add the corresponding label as a tensor

        return item


        

train_dataset = IMDBDataset(train_encodings, train_labels)
test_dataset = IMDBDataset(test_encodings, test_labels)
sample1 = train_dataset[0]
sample2 = test_dataset[0]
print(sample1.keys())
print(sample1["input_ids"][:10])
print(sample1["attention_mask"][:10])
print(sample1["labels"])
print(sample2.keys())
print(sample2["input_ids"][:10])
print(sample2["attention_mask"][:10])
print(sample2["labels"])


In [ ]:
# create DataLoader for train/test data
from torch.utils.data import DataLoader

# creat DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=6,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=12,
    shuffle=True
)

print("train_batch_size:", len(train_loader))
print("test_batch_size", len(test_loader))

# check the batch
batch = next(iter(train_loader))
print(batch.keys())
# check the shapes
print(batch["input_ids"].shape)
print(batch["attention_mask"].shape)
print(batch["labels"].shape)

In [ ]:
# build model - for BERT, yet has access to DistilBERT in SentimenClassifier
import torch.nn as nn
from transformers import AutoModel

class SentimentClassifier(nn.Module):

    def __init__(self, model_name="bert-base-uncased", num_labels=2, dropout_prob=0.3):
        super().__init__()

        self.model_name = model_name
        # load the bert model
        self.encoder = AutoModel.from_pretrained(model_name)
        # get the hidden size
        hidden_size = self.encoder.config.hidden_size
        # define the classification head
        self.dropout = nn.Dropout(dropout_prob)
        self.classifier = nn.Linear(hidden_size, num_labels)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        # BERT model can take token_type_ids, but DistilBERT cannot, using condition to handle this situation 
        if "distilbert" not in self.model_name:
            outputs = self.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids
            )
        else:
            outputs = self.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
        
        # extract the sentence representation from transofrmer outputs
        last_hidden_state = outputs.last_hidden_state  # [batch, length, hidden size]

        cls_embedding = last_hidden_state[:, 0, :]  # take the CLS embedding at position 0, shape becomes [batch, hidden size]

        # output classification score
        x = self.dropout(cls_embedding)  # apply dropout
        logits = self.classifier(x)  # classification output, shape [batch, num_labels]

        return logits

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("using device:", device)
bert_model = SentimentClassifier(model_name='bert-base-uncased').to(device)
batch = next(iter(train_loader))
input_ids = batch['input_ids'].to(device)
attention_mask = batch['attention_mask'].to(device)
token_type_ids = batch['token_type_ids'].to(device)
logits = bert_model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    token_type_ids=token_type_ids
)
print(logits.shape)
print(batch.keys())

## BERT Model

In [ ]:
# model training

epochs = 3
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(bert_model.parameters(), lr=2e-5)

# training history
bert_loss_history = []
bert_accuracy_history = []

for epoch in range(epochs):
    bert_model.train()

    total_loss = 0
    total_correct = 0
    total_samples = 0

    for step, batch in enumerate(train_loader, start=1):
        # move batch tensors to the device
        batch = {k: v.to(device) for k, v in batch.items()}
        input_ids = batch['input_ids']
        attention_mask = batch['attention_mask']
        token_type_ids = batch['token_type_ids']
        labels = batch['labels']

        # clear previous gradients
        optimizer.zero_grad()

        # forward
        logits = bert_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )

        # compute loss
        loss = loss_fn(logits,labels)

        # backward
        loss.backward()

        # update model parameters
        optimizer.step()

        # print training progress
        if step % 500 == 0:
            print(f"Epoch {epoch+1}, step {step}/{len(train_loader)}, batch_loss = {loss.item():.4f}")

        total_loss += loss.item()  # accumulate loss
        preds = torch.argmax(logits, dim=1)  # get predicted class
        total_correct += (preds == labels).sum().item()  # count correct predictions
        total_samples += labels.size(0)

    average_loss = total_loss / len(train_loader)
    accuracy = total_correct / total_samples
    bert_loss_history.append(average_loss)
    bert_accuracy_history.append(accuracy)
    # save parameters
    print(f"Epoch {epoch+1}: bert_average_loss = {average_loss:.4f}, accuracy = {accuracy:.4f}")
    torch.save(bert_model.state_dict(), f"bert_epoch{epoch+1}.pt")
    print(f"saved bert epoch {epoch+1}")

print("bert_loss_history =", bert_loss_history)
print("bert_accuracy_history =", bert_accuracy_history)

    

In [ ]:
# model evalation

# import saved parameters, best result occurs in epoch 3
bert_model.load_state_dict(torch.load("bert_epoch3.pt", map_location=device))
bert_model.eval()

loss_fn = nn.CrossEntropyLoss()

# initialize 
total_test_loss = 0
total_correct = 0
total_samples = 0

with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}

        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        token_type_ids = batch["token_type_ids"]
        labels = batch["labels"]

        # forward
        logits = bert_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
        # calculate loss
        loss = loss_fn(logits, labels)
        total_test_loss += loss.item()

        # get predictions, correct results, total samples
        preds = torch.argmax(logits, dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

# calculate average loss
bert_average_test_loss = total_test_loss / len(test_loader)
# calculate accuracy
bert_test_accuracy = total_correct / total_samples

print(f"BERT Test loss = {bert_average_test_loss:.4f}, BERT Test accuracy = {bert_test_accuracy:.4f}")


In [ ]:
# save BERT results
import json

with open("bert_results.json", "w", encoding="utf-8") as f:
    json.dump({
        "train_loss_history": bert_loss_history,
        "train_accuracy_history": bert_accuracy_history,
        "test_loss": bert_average_test_loss,
        "test_accuracy": bert_test_accuracy
    }, f, indent=2)

print("BERT results saved")


## DistilBERT 

In [ ]:
# build model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

distilbert_model = SentimentClassifier(model_name="distilbert-base-uncased").to(device)

# forward
batch = next(iter(train_loader))
batch = {k: v.to(device) for k, v in batch.items()}

logits = distilbert_model(
    input_ids=batch["input_ids"],
    attention_mask=batch["attention_mask"],
    token_type_ids=batch["token_type_ids"]
)

print(logits.shape)

In [ ]:
# train model
epochs = 3
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(distilbert_model.parameters(), lr=2e-5)

# training history
distilbert_loss_history = []
distilbert_accuracy_history = []

for epoch in range(epochs):
    distilbert_model.train()

    total_loss = 0
    total_correct = 0
    total_samples = 0

    for step, batch in enumerate(train_loader, start=1):
        batch = {k: v.to(device) for k, v in batch.items()}

        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        token_type_ids = batch["token_type_ids"]
        labels = batch["labels"]

        optimizer.zero_grad()

        # forward
        logits = distilbert_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )
        # loss, accuracy
        loss = loss_fn(logits, labels)

        loss.backward()
        optimizer.step()

        if step % 500 == 0:
            print(f"Epoch {epoch+1}, step {step}/{len(train_loader)}, batch_loss = {loss.item():.4f}")

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

    average_loss = total_loss / len(train_loader)
    accuracy = total_correct / total_samples
    distilbert_loss_history.append(average_loss)
    distilbert_accuracy_history.append(accuracy)

    print(f"Epoch {epoch+1}: average_loss = {average_loss:.4f}, accuracy = {accuracy:.4f}")
    torch.save(distilbert_model.state_dict(), f"distilbert_epoch{epoch+1}.pt")
    print(f"saved distilbert epoch {epoch+1}")

print("distilbert_loss_history =", distilbert_loss_history)
print("distilbert_accuracy_history =", distilbert_accuracy_history)

In [ ]:
# evaluation

# import parameters of epoch3, it's the best in the training
distilbert_model.load_state_dict(torch.load("distilbert_epoch3.pt", map_location=device))
distilbert_model.eval()
loss_fn = nn.CrossEntropyLoss()

# initialize
total_test_loss = 0
total_correct = 0
total_samples = 0

with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}

        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        token_type_ids = batch["token_type_ids"]
        labels = batch["labels"]

        # forward
        logits = distilbert_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )

        # cluculate loss
        loss = loss_fn(logits, labels)
        total_test_loss += loss.item()

        # get predictions, correct results, total samples
        preds = torch.argmax(logits, dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

# calculate average loss
distilbert_average_test_loss = total_test_loss / len(test_loader)
# calculate accuracy
distilbert_test_accuracy = total_correct / total_samples

print(f"DistilBERT Test loss = {distilbert_average_test_loss:.4f}, DistilBERT Test accuracy = {distilbert_test_accuracy:.4f}")


In [ ]:
# save DistilBERT results
import json

with open("distilbert_results.json", "w", encoding="utf-8") as f:
    json.dump({
        "train_loss_history": distilbert_loss_history,
        "train_accuracy_history": distilbert_accuracy_history,
        "test_loss": distilbert_average_test_loss,
        "test_accuracy": distilbert_test_accuracy
    }, f, indent=2)

print("DistilBERT saved")


## Random Baseline

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score

# fixed seed
np.random.seed(49)

# random predict: 0 or 1
random_preds = np.random.randint(0, 2, size=len(test_labels))

# calculate accuracy
random_accuracy = accuracy_score(test_labels, random_preds)

print(f"Random baseline accuracy = {random_accuracy:.4f}")

In [ ]:
# save random baseline result
with open("random_baseline_results.json", "w", encoding="utf-8") as f:
    json.dump({
        "test_accuracy": random_accuracy
    }, f, indent=2)

print("Random baseline result saved")


## mBERT

In [ ]:
# tokenize

# load the tokenizer for the mBERT model
mbert_tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")

# tokenize text data with the mBERT tokenizer
mbert_train_encodings = mbert_tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=512
)
mbert_test_encodings = mbert_tokenizer(
    test_texts,
    truncation=True,
    padding=True,
    max_length=512
)

print(len(mbert_train_encodings["input_ids"]))
print(len(mbert_train_encodings["attention_mask"]))
print(len(mbert_train_encodings["token_type_ids"]))


In [ ]:
# dataset and dataloader

# build dataset
mbert_train_dataset = IMDBDataset(mbert_train_encodings, train_labels)
mbert_test_dataset = IMDBDataset(mbert_test_encodings, test_labels)

# create DataLoader
mbert_train_loader = DataLoader(
    mbert_train_dataset,
    batch_size=6,
    shuffle=True
)

mbert_test_loader = DataLoader(
    mbert_test_dataset,
    batch_size=12,
    shuffle=True
)

print("mbert_train_batch_size:", len(mbert_train_loader))
print("mbert_test_batch_size", len(mbert_test_loader))

# check the batch
batch = next(iter(mbert_train_loader))
print(batch.keys())
# check the shapes
print(batch["input_ids"].shape)
print(batch["attention_mask"].shape)
print(batch["labels"].shape)


In [ ]:
# build mbert model

mbert_model = SentimentClassifier(model_name='bert-base-multilingual-cased').to(device)

mbert_batch = next(iter(mbert_train_loader))
mbert_batch = {k: v.to(device) for k, v in mbert_batch.items()}

input_ids = mbert_batch["input_ids"]
attention_mask = mbert_batch["attention_mask"]
token_type_ids = mbert_batch["token_type_ids"]

logits = mbert_model(
    input_ids=input_ids,
    attention_mask=attention_mask,
    token_type_ids=token_type_ids
)

print(logits.shape)

In [ ]:
# model training

epochs = 3
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(mbert_model.parameters(), lr=2e-5)

# training history
mbert_loss_history = []
mbert_accuracy_history = []

for epoch in range(epochs):
    mbert_model.train()

    total_loss = 0
    total_correct = 0
    total_samples = 0

    for step, batch in enumerate(mbert_train_loader, start=1):
        # move batch tensors to the device
        batch = {k: v.to(device) for k, v in batch.items()}
        input_ids = batch['input_ids']
        attention_mask = batch['attention_mask']
        token_type_ids = batch['token_type_ids']
        labels = batch['labels']

        # clear previous gradients
        optimizer.zero_grad()

        # forward
        logits = mbert_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )

        # compute loss
        loss = loss_fn(logits,labels)

        # backward
        loss.backward()

        # update model parameters
        optimizer.step()

        # print training progress
        if step % 500 == 0:
            print(f"Epoch {epoch+1}, step {step}/{len(mbert_train_loader)}, batch_loss = {loss.item():.4f}")

        total_loss += loss.item()  # accumulate loss
        preds = torch.argmax(logits, dim=1)  # get predicted class
        total_correct += (preds == labels).sum().item()  # count correct predictions
        total_samples += labels.size(0)

    average_loss = total_loss / len(mbert_train_loader)
    accuracy = total_correct / total_samples
    mbert_loss_history.append(average_loss)
    mbert_accuracy_history.append(accuracy)
    # save parameters
    print(f"Epoch {epoch+1}: mbert_average_loss = {average_loss:.4f}, accuracy = {accuracy:.4f}")
    torch.save(mbert_model.state_dict(), f"mbert_epoch{epoch+1}.pt")
    print(f"saved mbert epoch {epoch+1}")

print("mbert_loss_history =", mbert_loss_history)
print("mbert_accuracy_history =", mbert_accuracy_history)

    

In [ ]:
# evaluation

# import parameters of the best epoch after training
mbert_model.load_state_dict(torch.load("mbert_epoch3.pt", map_location=device))
mbert_model.eval()
loss_fn = nn.CrossEntropyLoss()

# initialize
total_test_loss = 0
total_correct = 0
total_samples = 0

with torch.no_grad():
    for batch in mbert_test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}

        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        token_type_ids = batch["token_type_ids"]
        labels = batch["labels"]

        # forward
        logits = mbert_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids
        )

        # calculate loss
        loss = loss_fn(logits, labels)
        total_test_loss += loss.item()

        # get predictions, correct results, total samples
        preds = torch.argmax(logits, dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

# calculate average loss and accuracy
mbert_average_test_loss = total_test_loss / len(mbert_test_loader)
mbert_test_accuracy = total_correct / total_samples

print(f"mbert Test loss = {mbert_average_test_loss:.4f}, mbert Test accuracy = {mbert_test_accuracy:.4f}")


In [ ]:
# save mBERT results
import json

with open("mbert_results.json", "w", encoding="utf-8") as f:
    json.dump({
        "train_loss_history": mbert_loss_history,
        "train_accuracy_history": mbert_accuracy_history,
        "test_loss": mbert_average_test_loss,
        "test_accuracy": mbert_test_accuracy
    }, f, indent=2)

print("mBERT results saved")


## ML - Logistic Regression with TF-IDF

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# build TF-IDF vectorizer
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    max_features=20000,
    ngram_range=(1,1)
)

# trainsfer text into TF-IDF features
X_train_tfidf = vectorizer.fit_transform(train_texts)
X_test_tfidf = vectorizer.transform(test_texts)

# build logistic regression classifier
ml_model = LogisticRegression(
    max_iter=1000,
    random_state=49
)

# train the model
ml_model.fit(X_train_tfidf, train_labels)

# prediction
train_preds = ml_model.predict(X_train_tfidf)
test_preds = ml_model.predict(X_test_tfidf)

# evaluation
ml_train_accuracy = accuracy_score(train_labels, train_preds)
ml_test_accuracy = accuracy_score(test_labels, test_preds)

print(f"Logistic Regression train accuracy = {ml_train_accuracy:.4f}")
print(f"Logistic Regression test accuracy = {ml_test_accuracy:.4f}")


In [ ]:
# save logistic regression results
import json

with open("logistic_regression_results.json", "w", encoding="utf-8") as f:
    json.dump({
        "train_accuracy": ml_train_accuracy,
        "test_accuracy": ml_test_accuracy
    }, f, indent=2)

print("Logistic regression results saved")


## Results

In [ ]:
import pandas as pd
import json

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

bert_results = load_json("bert_results.json")
distilbert_results = load_json("distilbert_results.json")
mbert_results = load_json("mbert_results.json")
random_baseline_results = load_json("random_baseline_results.json")
logistic_regression_results = load_json("logistic_regression_results.json")

# summarize epoch-level training results for transformer models
transformer_training_results = pd.DataFrame({
    "Model": ["BERT"] * len(bert_results["train_loss_history"]) + ["DistilBERT"] * len(distilbert_results["train_loss_history"]) + ["mBERT"] * len(mbert_results["train_loss_history"]),
    "Epoch": list(range(1, len(bert_results["train_loss_history"]) + 1)) + list(range(1, len(distilbert_results["train_loss_history"]) + 1)) + list(range(1, len(mbert_results["train_loss_history"]) + 1)),
    "Train Loss": bert_results["train_loss_history"] + distilbert_results["train_loss_history"] + mbert_results["train_loss_history"],
    "Train Accuracy": bert_results["train_accuracy_history"] + distilbert_results["train_accuracy_history"] + mbert_results["train_accuracy_history"],
})

transformer_training_results["Train Loss"] = transformer_training_results["Train Loss"].round(4)
transformer_training_results["Train Accuracy"] = transformer_training_results["Train Accuracy"].round(4)

print("Transformer training results by epoch")
display(transformer_training_results)

# summarize final train/test results for all models
final_results = pd.DataFrame([
    {
        "Model": "Random Baseline",
        "Best Epoch": "-",
        "Final Train Accuracy": "-",
        "Final Test Loss": "-",
        "Final Test Accuracy": round(random_baseline_results["test_accuracy"], 4),
    },
    {
        "Model": "Logistic Regression (TF-IDF)",
        "Best Epoch": "-",
        "Final Train Accuracy": round(logistic_regression_results["train_accuracy"], 4),
        "Final Test Loss": "-",
        "Final Test Accuracy": round(logistic_regression_results["test_accuracy"], 4),
    },
    {
        "Model": "BERT",
        "Best Epoch": int(pd.Series(bert_results["train_accuracy_history"]).idxmax() + 1),
        "Final Train Accuracy": round(max(bert_results["train_accuracy_history"]), 4),
        "Final Test Loss": round(bert_results["test_loss"], 4),
        "Final Test Accuracy": round(bert_results["test_accuracy"], 4),
    },
    {
        "Model": "DistilBERT",
        "Best Epoch": int(pd.Series(distilbert_results["train_accuracy_history"]).idxmax() + 1),
        "Final Train Accuracy": round(max(distilbert_results["train_accuracy_history"]), 4),
        "Final Test Loss": round(distilbert_results["test_loss"], 4),
        "Final Test Accuracy": round(distilbert_results["test_accuracy"], 4),
    },
    {
        "Model": "mBERT",
        "Best Epoch": int(pd.Series(mbert_results["train_accuracy_history"]).idxmax() + 1),
        "Final Train Accuracy": round(max(mbert_results["train_accuracy_history"]), 4),
        "Final Test Loss": round(mbert_results["test_loss"], 4),
        "Final Test Accuracy": round(mbert_results["test_accuracy"], 4),
    },
])

print("Final comparison across all models")
display(final_results)

best_model = final_results.loc[final_results["Final Test Accuracy"].astype(float).idxmax(), "Model"]
print(f"Best test accuracy: {best_model}")


In [ ]:
import matplotlib.pyplot as plt

# epoch index
bert_epochs = range(1, len(bert_results["train_loss_history"]) + 1)
distilbert_epochs = range(1, len(distilbert_results["train_loss_history"]) + 1)
mbert_epochs = range(1, len(mbert_results["train_loss_history"]) + 1)

# plot training loss and accuracy side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(bert_epochs, bert_results["train_loss_history"], marker="o", label="BERT")
axes[0].plot(distilbert_epochs, distilbert_results["train_loss_history"], marker="o", label="DistilBERT")
axes[0].plot(mbert_epochs, mbert_results["train_loss_history"], marker="o", label="mBERT")
axes[0].set_title("Training Loss by Epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_xticks(list(bert_epochs))
axes[0].legend()
axes[0].grid(True)

axes[1].plot(bert_epochs, bert_results["train_accuracy_history"], marker="o", label="BERT")
axes[1].plot(distilbert_epochs, distilbert_results["train_accuracy_history"], marker="o", label="DistilBERT")
axes[1].plot(mbert_epochs, mbert_results["train_accuracy_history"], marker="o", label="mBERT")
axes[1].set_title("Training Accuracy by Epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].set_xticks(list(bert_epochs))
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## Analysis

Before going through the results, we first briefly explain the evaluation metric. In this experiment, we mainly use accuracy to measure model performance. Since the dataset is about movie reviews, it's relatively balanced, accuracy provides a reasonable results of how well the model perform.

We start by examining the training process of the transformer-based models: from the training curves, all models show a clear and stable learning trend: the loss decreases while the accuracy increases over epochs. This suggests that all models are able to learn the sentiment classification task effictively.

Among the transformer models, BERT reaches the lowest training loss and highest accracy. DistilBERT performs slightly lower, this is expected, as DistilBERT is a smaller and more efficient version of BERT. The multilingual model (mBERT), on the other hand, performs noticeably worse, with higher loss and lower accuracy during training.

Looking at the final results, the random baseline achieves an accuracy close to 0.5, which is expected for a binary classification task. This confirms that better performance requires actual learning.

The machine learning model (Logistic Regression with TF-IDF) performs significantly better than the random baseline. This shows that even simple models with good feature representations can capture useful patterns in the data.

However, the transformer-based model achieve the best performance overall. All three versions of BERT outperform the machine learning baseline, indicates that the advantage of contextual representations over traditional approaches.

Comparing BERT and multilingual BERT, BERT consistently outperforms mBERT. This suggests that models trained specifically for one language are more effective than multilingual models.

Overall, the results show that transformer-based models are highly effctive for sentiment classification, while simpler models can still offer competitive performance with much lower coomputational cost.

## Discussions

In this assignment, one of the main challenges was handling the data processing for different transformer models. Although the overall pipeline is similar, different models (BERT, DistilBERT, mBERT) require different input formats. Adjusting the data processing for each model took a lot of time. In paticular, the mBERT introduced additional difficultie, and some training issues were caused by mismatched inputs.

Another challange was managing GPU memory. Due to the relatively large size of the dataset and the transformer models, training often require careful memory handling. This included reducing batch size and clearing GPU cache completely, which was not something I had much experience with before.

In addition, I learned the importance of saving model parameters and experimental results during training. In earlier runs, I did not save intermediate results, which forced me to rerun experiments multiple times. This experience taught me give attention on proper experiment management.

From a learning perspective, this assignment was very valuable. I gained a much better understanding of the full training pipeline, from data loading and preprocessing to moedl training and evaluation. It alsohelped me develop a clearer intuition about how different models behave and how to analyze their performance.

The most time-consuming part of the assignment was training the models. Since the dataset is quite large, running multiple experiments required a lot of time. In addition, debugging issues to data processing and model inputs also consumed amount of time.